<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: Una Flota de Buses que no Deja de Crecer 🚌
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 00
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/homeworks/00_Fundamentos_Big_Data_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 00: Fundamentos de Big Data** ([README del módulo](../00%20-%20Fundamentos%20de%20Big%20Data/README.md)) con un problema **nuevo**: la telemetría de una **flota ficticia de 180 buses intermunicipales** (lecturas GPS, alertas de frenado, incidentes y quejas de pasajeros).

Al terminar serás capaz de:

1. **Dimensionar el volumen** de una fuente de datos (unidades SI frente a binarias, replicación, crecimiento compuesto) y aplicar la **ley de Amdahl** para decidir cuántos procesadores valen la pena.
2. Agrupar un flujo de eventos en **ventanas de tiempo** y distinguir **tiempo de evento** de **tiempo de llegada**, midiendo qué cuesta la **marca de agua** (*watermark*) que tolera eventos tardíos.
3. **Armonizar la variedad**: CSV, JSON de esquema variable y texto libre llevados a un esquema común, con **cuarentena** de lo que no se puede leer.
4. Recorrer el **ciclo de vida** de un dato: validar con reglas, medir su calidad, separar por **retención**, **seudonimizar** y atender un **derecho de supresión** en todas las zonas.

> 📚 **Repasa si lo necesitas:** [Introducción y las «V»](../00%20-%20Fundamentos%20de%20Big%20Data/00_Introduccion_y_Las_V_del_Big_Data.ipynb) · [Tipos de datos](../00%20-%20Fundamentos%20de%20Big%20Data/02_Tipos_de_Datos_Estructurados_No_Estructurados_Semiestructurados.ipynb) · [Ciclo de vida de los datos](../00%20-%20Fundamentos%20de%20Big%20Data/03_Ciclo_de_Vida_de_los_Datos.ipynb).

---
## Instrucciones 📋

1. **Resuelve de forma autónoma:** en cada reto completa la celda de código marcada con `# TODO:`. Este cuaderno **no trae soluciones**.
2. **Autoverificación:** debajo de cada reto hay una celda con `assert`. Si tu solución es correcta imprime ✅; si no, el mensaje te dice qué revisar. No la modifiques.
3. **Puntaje:** cada reto indica su puntaje sugerido; el total del taller suma **100 puntos**.
4. Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin, sin errores y **sin `# TODO` pendientes**.
5. Todo funciona **sin internet y sin archivos externos** (datos sintéticos con semilla fija; los datos y la flota son **inventados**). Bibliotecas: `numpy` y `pandas` más la biblioteca estándar (`json`, `re`, `hashlib`, `io`). Cada celda corre en menos de un segundo.

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos sintéticos
# ============================================================
import hashlib
import io
import json
import math
import re

import numpy as np
import pandas as pd

SEMILLA = 7
BUSES = [f"B-{i:03d}" for i in range(1, 13)]

# --- Constantes del Reto 1 (flota ficticia) ---
N_BUSES, INTERVALO_S, BYTES_LECTURA, REPLICACION = 180, 10, 220, 3
CAPACIDAD_BYTES = 500 * 10**9      # 500 GB (SI) de disco disponible
CRECIMIENTO = 0.04                 # la flota crece 4 % cada mes
S_SERIAL = 0.12                    # fracción secuencial del trabajo nocturno de la flota


def generar_eventos(semilla=SEMILLA, n=3000, duracion=600):
    """Alertas de frenado brusco: bus, tiempo del evento y tiempo de llegada (s)."""
    rng = np.random.default_rng(semilla)
    t_ev = np.sort(rng.uniform(0, duracion, n))
    retraso = rng.uniform(0, 3, n) + (rng.random(n) < 0.08) * rng.uniform(20, 90, n)   # 8 % llega muy tarde
    return pd.DataFrame({"bus": rng.choice(BUSES, n), "t_evento": t_ev, "t_llegada": t_ev + retraso})


def generar_fuentes(semilla=SEMILLA):
    """Tres fuentes heterogéneas con defectos sembrados: CSV de GPS, JSON por líneas y quejas en texto."""
    rng = np.random.default_rng(semilla)
    t0 = pd.Timestamp("2025-03-10 08:00:00")
    # 1) CSV de GPS: 300 lecturas, 15 con defectos
    n = 300
    bus = [str(b) for b in rng.choice(BUSES, n)]
    vel = [str(v) for v in np.round(rng.uniform(0, 80, n), 1)]
    lat = np.round(rng.uniform(5.40, 5.90, n), 5)
    lon = np.round(rng.uniform(-73.60, -73.20, n), 5)
    malos = rng.choice(n, 15, replace=False)
    for j in malos[:6]:
        vel[j] = "n/d"
    for j in malos[6:10]:
        bus[j] = ""
    for j in malos[10:]:
        vel[j] = str(round(float(rng.uniform(150, 220)), 1))
    filas = ["bus,ts,lat,lon,vel_kmh"] + [
        f"{bus[i]},{(t0 + pd.Timedelta(seconds=10 * i)).isoformat()},{lat[i]},{lon[i]},{vel[i]}" for i in range(n)]
    # 2) JSON por líneas: 120 incidentes de esquema variable, 3 líneas truncadas
    lineas = []
    for i in range(120):
        d = {"id": i + 1, "bus": str(rng.choice(BUSES)), "ts": (t0 + pd.Timedelta(minutes=7 * i)).isoformat(),
             "tipo": str(rng.choice(["frenado", "falla_motor", "pinchazo", "retraso"]))}
        if rng.random() < 0.4:
            d["detalle"] = {"gravedad": int(rng.integers(1, 6)), "causa": str(rng.choice(["clima", "mecanica", "trafico"]))}
        if rng.random() < 0.3:
            d["pasajeros_afectados"] = int(rng.integers(1, 40))
        lineas.append(json.dumps(d, ensure_ascii=False))
    for j in rng.choice(120, 3, replace=False):
        lineas[j] = lineas[j][:-7]
    # 3) Quejas en texto libre: «fecha | bus | comentario»; 8 sin identificar el bus
    plantillas = ["llegó {m} min tarde a la parada", "salió {m} minutos después de lo programado",
                  "conductor brusco, retraso de {m} min", "el bus iba muy lleno", "música muy alta",
                  "no se detuvo en la parada"]
    sin_bus = set(int(k) for k in rng.choice(80, 8, replace=False))
    quejas = []
    for i in range(80):
        f = (t0 + pd.Timedelta(minutes=13 * i)).strftime("%Y-%m-%d %H:%M")
        txt = plantillas[int(rng.integers(len(plantillas)))].format(m=int(rng.integers(5, 60)))
        quejas.append(f"{f} | {'sin bus' if i in sin_bus else str(rng.choice(BUSES))} | {txt}")
    return "\n".join(filas), "\n".join(lineas), "\n".join(quejas)


def generar_viajes(semilla=SEMILLA, n=500):
    """Viajes con tarjeta: 12 id repetidos, 9 tarjetas nulas y 7 valores no positivos sembrados."""
    rng = np.random.default_rng(semilla)
    df = pd.DataFrame({"id_viaje": np.arange(1, n + 1),
                       "tarjeta": [f"T-{k:04d}" for k in rng.integers(1, 61, n)],
                       "bus": rng.choice(BUSES, n),
                       "fecha": pd.Timestamp("2023-07-01") + pd.to_timedelta(rng.integers(0, 730, n), unit="D"),
                       "valor_cop": rng.choice([2800, 3200, 4500], n)})
    idx = rng.permutation(n)
    df.loc[idx[:9], "tarjeta"] = None
    df.loc[idx[9:16], "valor_cop"] = rng.choice([0, -3200], 7)
    return pd.concat([df, df.loc[idx[16:28]]], ignore_index=True)


print("Constantes de la flota:", N_BUSES, "buses | 1 lectura cada", INTERVALO_S, "s |", BYTES_LECTURA, "bytes por lectura")

---
## Reto 1: Cuánto pesa una flota que no deja de crecer 📦

**Contexto:** cada uno de los 180 buses envía una lectura GPS de 220 bytes cada 10 segundos, las 24 horas. Los datos se guardan con **replicación 3** en un servidor con 500 GB (SI) libres, y la flota **crece 4 % cada mes**. Además, un trabajo nocturno de limpieza tiene un 12 % de trabajo estrictamente secuencial.

**Enunciado**
1. Implementa `bytes_dia(n_buses, intervalo_s, bytes_lectura)`: bytes que genera la flota en 24 h.
2. `tb_anio`: terabytes **SI** ($10^{12}$ bytes) que se almacenan en un año de 365 días **contando la replicación**; `tib_anio`: lo mismo en tebibytes ($2^{40}$ bytes).
3. `meses_duplicar`: meses que tarda en duplicarse la flota (y con ella el volumen diario) con crecimiento compuesto de `CRECIMIENTO` mensual.
4. `mes_llenado`: primer mes $m = 1, 2, \dots$ en que lo **acumulado** alcanza `CAPACIDAD_BYTES`. Supón meses de 30 días; en el mes $k$ la flota tiene `N_BUSES * (1 + CRECIMIENTO) ** (k - 1)` buses y se almacena con replicación.
5. Ley de Amdahl: `speedup(p)` $= \dfrac{1}{s + \frac{1 - s}{p}}$ con $s$ = `S_SERIAL`; `techo` (límite cuando $p \to \infty$) y `p_80`: menor número de procesadores que logra al menos el **80 %** del techo.

**Criterios de aceptación:** tus cifras coinciden con un cálculo independiente; `tib_anio < tb_anio`; y `speedup(p_80)` llega al 80 % del techo pero `speedup(p_80 - 1)` no.

**Puntaje sugerido:** 20 puntos.

In [ ]:
def bytes_dia(n_buses, intervalo_s, bytes_lectura):
    # TODO: bytes que genera la flota en 24 h (una lectura por bus cada `intervalo_s` segundos)
    return None


# TODO: completa
tb_anio = None         # TB (SI) por año de 365 días, con replicación
tib_anio = None        # TiB por año (misma cantidad de bytes)
meses_duplicar = None  # meses para duplicar la flota con CRECIMIENTO mensual compuesto

mes_llenado = None     # primer mes en que lo acumulado alcanza CAPACIDAD_BYTES


def speedup(p):
    # TODO: ley de Amdahl con fracción secuencial S_SERIAL
    return None


techo = None           # límite de la aceleración cuando p → ∞
p_80 = None            # menor p con speedup(p) >= 0.8 * techo

In [ ]:
# Autoverificación del Reto 1 (no modificar)
assert bytes_dia(1, 1, 1) == 86400 and bytes_dia(180, 10, 220) == 342_144_000, "bytes_dia = n_buses × (segundos por día / intervalo_s) × bytes_lectura (86 400 s por día)."
_b = 342_144_000 * 3 * 365
assert tb_anio is not None and abs(tb_anio - _b / 1e12) < 1e-9, "tb_anio = bytes por día × REPLICACION × 365 / 10**12."
assert tib_anio is not None and abs(tib_anio - _b / 2**40) < 1e-9 and tib_anio < tb_anio, "tib_anio usa 2**40 bytes por TiB; con los mismos bytes debe salir menor que tb_anio."
assert meses_duplicar is not None and abs(meses_duplicar - math.log(2) / math.log(1.04)) < 1e-9, "Con crecimiento compuesto: (1 + CRECIMIENTO) ** m = 2, es decir m = ln 2 / ln(1 + CRECIMIENTO)."
_acum, _m = 0.0, 0
while _acum < CAPACIDAD_BYTES:
    _m += 1
    _acum += 342_144_000 * 1.04 ** (_m - 1) * 30 * 3
assert mes_llenado == _m, f"mes_llenado debe ser el primer mes en que lo ACUMULADO (meses 1..m, con la flota creciendo y la replicación) alcanza la capacidad; esperaba {_m}."
assert abs(speedup(1) - 1) < 1e-12 and abs(speedup(10**9) - 1 / 0.12) < 1e-6, "speedup(1) debe ser 1 y, para p enorme, tender a 1 / S_SERIAL."
assert techo is not None and abs(techo - 1 / S_SERIAL) < 1e-12, "techo = 1 / S_SERIAL."
assert isinstance(p_80, (int, np.integer)) and speedup(p_80) >= 0.8 * techo > speedup(p_80 - 1), "p_80 debe ser el MENOR entero p con speedup(p) >= 0.8 * techo."
print(f"✅ Reto 1 superado: {tb_anio:.3f} TB/año ({tib_anio:.3f} TiB) | duplicar en {meses_duplicar:.1f} meses | se llena el mes {mes_llenado} | techo {techo:.2f}× y p_80 = {p_80}")

---
## Reto 2: Ventanas, retrasos y el precio de esperar ⏱️

**Contexto:** la flota emite alertas de **frenado brusco**. Cada evento tiene un **tiempo de evento** (`t_evento`, cuándo ocurrió) y un **tiempo de llegada** (`t_llegada`, cuándo lo recibe el sistema). El 8 % llega muy tarde por mala cobertura. Quieres contar alertas por **ventanas *tumbling*** de `ANCHO = 60` segundos.

**Enunciado**
1. `ventana`: inicio de la ventana de cada evento según `t_evento` (múltiplo de `ANCHO`).
2. `conteo_evento` y `conteo_llegada`: número de eventos por inicio de ventana, agrupando por **tiempo de evento** y por **tiempo de llegada** (series).
3. `n_ventanas_distintas`: en cuántas ventanas (del conjunto unión de ambas series; un faltante cuenta como 0) los dos conteos **difieren**.
4. `aceptados(eventos, L)`: con una **marca de agua** de tolerancia `L` segundos, la ventana de un evento se cierra en `fin + L` (con `fin` = inicio + `ANCHO`); se acepta el evento solo si `t_llegada <= fin + L`. Devuelve el `DataFrame` de eventos aceptados.
5. `perdidos`: diccionario `{L: eventos perdidos}` para cada `L` en `CANDIDATOS_L`, y `L_sin_perdidas`: el menor `L` candidato sin pérdidas.
6. `retraso_p95`: percentil 95 del retraso `t_llegada - t_evento`.

**Criterios de aceptación:** las cifras coinciden con un cálculo independiente; las pérdidas **no aumentan** al subir `L`; `L = 0` pierde eventos; y con el `L` elegido ya no se pierde ninguno (pero el resultado de cada ventana tarda `L` segundos más en publicarse).

**Puntaje sugerido:** 20 puntos.

In [ ]:
eventos = generar_eventos()
ANCHO = 60                         # segundos por ventana
CANDIDATOS_L = [0, 5, 15, 60, 120]  # tolerancias a evaluar (segundos)

# TODO: completa
ventana = None                # inicio de ventana (múltiplo de ANCHO) de cada evento, por t_evento
conteo_evento = None          # Serie: eventos por inicio de ventana, agrupando por TIEMPO DE EVENTO
conteo_llegada = None         # Serie: eventos por inicio de ventana, agrupando por TIEMPO DE LLEGADA
n_ventanas_distintas = None   # ventanas (unión de ambas) donde los conteos difieren


def aceptados(eventos, L):
    # TODO: devuelve los eventos con t_llegada <= (inicio de su ventana + ANCHO) + L
    return None


perdidos = None               # {L: nº de eventos perdidos} para L en CANDIDATOS_L
L_sin_perdidas = None         # menor L candidato sin pérdidas
retraso_p95 = None            # percentil 95 de t_llegada - t_evento

In [ ]:
# Autoverificación del Reto 2 (no modificar)
_t, _l = eventos["t_evento"].to_numpy(), eventos["t_llegada"].to_numpy()
_wi, _wl = np.floor(_t / ANCHO) * ANCHO, np.floor(_l / ANCHO) * ANCHO
assert ventana is not None and np.allclose(np.asarray(ventana, dtype=float), _wi), "ventana = (t_evento // ANCHO) * ANCHO: el inicio de la ventana de cada evento."
_ce = pd.Series(_wi).value_counts().to_dict()
_cl = pd.Series(_wl).value_counts().to_dict()
assert conteo_evento is not None and {int(k): int(v) for k, v in conteo_evento.items()} == {int(k): v for k, v in _ce.items()}, "conteo_evento: agrupa por el inicio de ventana calculado con t_evento y cuenta filas."
assert conteo_llegada is not None and {int(k): int(v) for k, v in conteo_llegada.items()} == {int(k): v for k, v in _cl.items()}, "conteo_llegada: igual, pero con el inicio de ventana calculado con t_llegada."
assert sum(conteo_evento) == sum(conteo_llegada) == len(eventos), "Ambas formas de contar deben sumar el total de eventos."
_dist = sum(1 for w in set(_ce) | set(_cl) if _ce.get(w, 0) != _cl.get(w, 0))
assert n_ventanas_distintas == _dist and _dist > 0, f"n_ventanas_distintas: ventanas (unión de ambas) con conteos distintos, contando faltantes como 0; esperaba {_dist}."
for _L in (0, 30, 120):
    _esp = int(np.sum(_l <= _wi + ANCHO + _L))
    _ac = aceptados(eventos, _L)
    assert _ac is not None and len(_ac) == _esp, f"aceptados(eventos, {_L}) debe conservar los eventos con t_llegada <= inicio + ANCHO + L; esperaba {_esp} filas."
assert isinstance(perdidos, dict) and set(perdidos) == set(CANDIDATOS_L), "perdidos debe ser un diccionario con una clave por cada L de CANDIDATOS_L."
assert all(perdidos[L] == len(eventos) - int(np.sum(_l <= _wi + ANCHO + L)) for L in CANDIDATOS_L), "perdidos[L] = total de eventos - eventos aceptados con esa tolerancia."
_p = [perdidos[L] for L in sorted(CANDIDATOS_L)]
assert all(a >= b for a, b in zip(_p, _p[1:])) and perdidos[0] > 0, "Las pérdidas no deben aumentar al subir L, y con L = 0 sí se pierden eventos."
assert L_sin_perdidas == min(L for L in CANDIDATOS_L if perdidos[L] == 0), "L_sin_perdidas es el menor L candidato con cero pérdidas."
assert retraso_p95 is not None and abs(retraso_p95 - np.percentile(_l - _t, 95)) < 1e-9, "retraso_p95 = np.percentile(t_llegada - t_evento, 95)."
print(f"✅ Reto 2 superado: {n_ventanas_distintas} ventanas cambian según el reloj | pérdidas por L = {perdidos} | sin pérdidas desde L = {L_sin_perdidas} s (p95 del retraso = {retraso_p95:.1f} s)")

---
## Reto 3: Tres fuentes, un solo esquema 🧩

**Contexto:** la empresa recibe datos de tres formas: un **CSV** de lecturas GPS (estructurado), **JSON por líneas** de incidentes (semiestructurado, de esquema variable) y **quejas en texto libre** (no estructurado). Las tres traen defectos. `csv_gps`, `jsonl` y `texto_quejas` ya están cargadas como cadenas.

**Enunciado**
1. **GPS:** lee el CSV (`io.StringIO`). Una lectura es **válida** si `bus` no está vacío y `vel_kmh` es numérica y está entre 0 y 120. Guarda las válidas en `gps` (con `vel_kmh` numérica) y cuenta el resto en `n_cuarentena_gps`.
2. **Incidentes:** implementa `leer_jsonl(texto)` → `(registros_validos, n_malas)` (una línea que no es JSON válido va a cuarentena). Con ellos arma `incidentes = pd.json_normalize(registros)`: los objetos anidados se aplanan (`detalle.gravedad`) y lo ausente queda como `NaN`.
3. **Quejas:** implementa `extraer_queja(linea)` → `{"bus": "B-017", "minutos": 25}` o `None` si la línea no identifica el bus (formato `fecha | bus | comentario`; `minutos` es `None` si el comentario no menciona «N min»). Arma `quejas` (`DataFrame` con `bus` y `minutos`) y `n_cuarentena_quejas`.
4. **Unificación:** `resumen_bus`: `DataFrame` indexado por bus (todos los que aparecen en alguna fuente, ordenados) con las columnas enteras `n_gps`, `n_incidentes` y `n_quejas` (0 si no aparece).

**Criterios de aceptación:** en cada fuente **válidas + cuarentena = líneas de entrada** (conservación); los conteos coinciden con un recorrido independiente; y los totales de `resumen_bus` suman lo que tiene cada tabla.

**Puntaje sugerido:** 20 puntos.

In [ ]:
csv_gps, jsonl, texto_quejas = generar_fuentes()

# TODO: completa
gps = None                   # DataFrame con bus, ts, lat, lon, vel_kmh (solo lecturas válidas)
n_cuarentena_gps = None


def leer_jsonl(texto):
    # TODO: devuelve (lista de dicts válidos, número de líneas que no son JSON válido)
    return None


registros, n_cuarentena_inc = None, None   # resultado de leer_jsonl(jsonl)
incidentes = None                          # pd.json_normalize(registros)


def extraer_queja(linea):
    # TODO: {"bus": ..., "minutos": ...} o None si la línea no identifica el bus
    return None


quejas = None                # DataFrame con las columnas bus y minutos
n_cuarentena_quejas = None

resumen_bus = None           # índice = bus; columnas n_gps, n_incidentes, n_quejas

In [ ]:
# Autoverificación del Reto 3 (no modificar)
_lin_gps = csv_gps.splitlines()[1:]
_v = 0
for _ln in _lin_gps:
    _b, _ts, _la, _lo, _ve = _ln.split(",")
    try:
        _x = float(_ve)
    except ValueError:
        continue
    _v += bool(_b) and 0 <= _x <= 120
assert gps is not None and len(gps) == _v and n_cuarentena_gps == len(_lin_gps) - _v, f"GPS: debían quedar {_v} lecturas válidas y {len(_lin_gps) - _v} en cuarentena (bus vacío, velocidad no numérica o fuera de 0-120)."
assert pd.api.types.is_numeric_dtype(gps["vel_kmh"]) and gps["vel_kmh"].between(0, 120).all() and (gps["bus"] != "").all(), "En gps, vel_kmh debe ser numérica (convierte con pd.to_numeric) y estar entre 0 y 120."
_t = leer_jsonl('{"a": 1}\n{"a": \n{"b": {"c": 2}}')
assert _t is not None and _t[0] == [{"a": 1}, {"b": {"c": 2}}] and _t[1] == 1, "leer_jsonl debe devolver ([{'a': 1}, {'b': {'c': 2}}], 1) para el ejemplo de tres líneas con una malformada."
_val = 0
for _ln in jsonl.splitlines():
    try:
        json.loads(_ln)
        _val += 1
    except json.JSONDecodeError:
        pass
assert len(registros) == _val == len(incidentes) and n_cuarentena_inc == 120 - _val == 3, "Incidentes: 120 líneas = válidas + 3 malformadas; 'incidentes' debe tener una fila por registro válido."
_sin_pas = sum("pasajeros_afectados" not in json.loads(l) for l in jsonl.splitlines() if l.endswith("}") and l.count("{") == l.count("}"))
assert {"detalle.gravedad", "detalle.causa", "pasajeros_afectados"} <= set(incidentes.columns), "json_normalize debe aplanar el objeto anidado en columnas como 'detalle.gravedad'."
assert int(incidentes["pasajeros_afectados"].isna().sum()) == _sin_pas, "Los registros sin 'pasajeros_afectados' deben quedar como NaN (ausente, no cero)."
assert extraer_queja("2025-03-14 08:12 | B-017 | llegó 25 min tarde a Duitama") == {"bus": "B-017", "minutos": 25}, "extraer_queja: bus con la expresión B-\\d{3} y minutos con (\\d+)\\s*min."
_e = extraer_queja("2025-03-14 08:12 | B-003 | música muy alta")
assert _e is not None and _e["bus"] == "B-003" and _e["minutos"] is None, "Si el comentario no menciona minutos, 'minutos' debe ser None."
assert extraer_queja("2025-03-14 08:12 | sin bus | llegó 5 min tarde") is None, "Una línea que no identifica el bus debe devolver None (cuarentena)."
_lin_q = texto_quejas.splitlines()
assert len(quejas) + n_cuarentena_quejas == len(_lin_q) == 80 and n_cuarentena_quejas == 8, "Quejas: 80 líneas = 72 con bus + 8 sin bus."
assert list(quejas.columns) == ["bus", "minutos"], "quejas debe tener exactamente las columnas bus y minutos."
_ms = sum(bool(re.search(r"\d+\s*min", l.split("|")[2])) for l in _lin_q if "sin bus" not in l)
assert int(quejas["minutos"].notna().sum()) == _ms, "El número de quejas con 'minutos' no coincide con las que mencionan «N min»."
assert resumen_bus is not None and list(resumen_bus.columns) == ["n_gps", "n_incidentes", "n_quejas"], "resumen_bus debe tener las columnas n_gps, n_incidentes y n_quejas."
assert list(resumen_bus.index) == sorted(resumen_bus.index) and resumen_bus.index.is_unique, "El índice de resumen_bus debe ser único y estar ordenado."
assert resumen_bus.to_numpy().dtype.kind == "i", "Los conteos deben ser enteros (rellena con 0 lo que no aparece y convierte con astype(int))."
assert tuple(resumen_bus.sum()) == (len(gps), len(incidentes), len(quejas)), "Los totales de resumen_bus deben sumar el número de filas de gps, incidentes y quejas."
print(f"✅ Reto 3 superado: GPS {len(gps)}+{n_cuarentena_gps}, incidentes {len(incidentes)}+{n_cuarentena_inc}, quejas {len(quejas)}+{n_cuarentena_quejas} (válidas + cuarentena = entrada)")

---
## Reto 4: De la tarjeta al archivo, con derecho a olvido 🗂️

**Contexto:** el sistema de recaudo registra los viajes pagados con tarjeta (`generar_viajes()`): `id_viaje`, `tarjeta`, `bus`, `fecha` y `valor_cop`. Llegan con defectos: ids repetidos, tarjetas nulas y valores no positivos. Vas a llevarlos por las **zonas** del ciclo de vida (cruda → curada → archivo, más una **cuarentena**) y a atender la petición de una usuaria que pide que se **borren sus datos**.

**Enunciado**
1. `validar(crudo)` → `(curada, cuarentena)`. Motivo de cuarentena, en este orden de prioridad: `"duplicado"` (id repetido: se conserva la primera aparición), `"tarjeta_nula"`, `"valor_invalido"` (`valor_cop <= 0`). `cuarentena` lleva una columna extra `motivo`.
2. `calidad(df)` → diccionario con `completitud` (fracción de `tarjeta` no nula), `unicidad` (ids distintos / filas) y `validez` (fracción con `valor_cop > 0`). Calcula `q_crudo` y `q_curada`.
3. `separar_retencion(df, hoy, dias)` → `(vigente, archivo)`: va al `archivo` lo anterior a `hoy - dias` días. Usa `HOY` y `DIAS_RETENCION`.
4. `recibo`: `{"entran": ..., "curada": ..., "cuarentena": ...}` y verifica que **entran = curada + cuarentena**.
5. `pseudonimo(tarjeta, sal)`: primeros 16 caracteres hexadecimales de `sha256(sal + tarjeta)`. Obtén `ps_a1`, `ps_a2` (misma sal, dos veces) y `ps_b` (otra sal).
6. `suprimir(zonas, tarjeta)` → `(zonas_nuevas, n_borradas)`: elimina esa tarjeta de **todas** las zonas (`cruda`, `vigente`, `archivo`, `cuarentena`).

**Criterios de aceptación:** conservación de filas en cada paso; la calidad de la zona curada es 1.0 en las tres dimensiones y la de la cruda es menor; el seudónimo es determinista y depende de la sal; y tras suprimir **no queda ninguna fila** de la tarjeta en ninguna zona, sin tocar las demás.

**Puntaje sugerido:** 25 puntos.

In [ ]:
crudo = generar_viajes()
HOY, DIAS_RETENCION = pd.Timestamp("2025-07-01"), 540


def validar(crudo):
    # TODO: devuelve (curada, cuarentena); cuarentena con columna `motivo`
    return None


def calidad(df):
    # TODO: {"completitud": ..., "unicidad": ..., "validez": ...}
    return None


def separar_retencion(df, hoy, dias):
    # TODO: devuelve (vigente, archivo)
    return None


def pseudonimo(tarjeta, sal):
    # TODO: sha256(sal + tarjeta), primeros 16 caracteres hexadecimales
    return None


def suprimir(zonas, tarjeta):
    # TODO: devuelve (zonas_nuevas, n_borradas)
    return None


# TODO: completa
curada = cuarentena = None
q_crudo = q_curada = None
vigente = archivo = None
recibo = None                      # {"entran": ..., "curada": ..., "cuarentena": ...}
ps_a1 = ps_a2 = ps_b = None        # pseudonimo("T-0007", "sal-A") dos veces y pseudonimo("T-0007", "sal-B")

TARJETA = "T-0007"
zonas = None                       # {"cruda": crudo, "vigente": vigente, "archivo": archivo, "cuarentena": cuarentena}
zonas_nuevas = n_borradas = None

In [ ]:
# Autoverificación del Reto 4 (no modificar)
_dup = crudo["id_viaje"].duplicated()
_nul = ~_dup & crudo["tarjeta"].isna()
_val = ~_dup & ~_nul & (crudo["valor_cop"] <= 0)
assert curada is not None and cuarentena is not None and "motivo" in cuarentena.columns, "validar debe devolver (curada, cuarentena) y la cuarentena debe llevar la columna 'motivo'."
assert len(curada) == len(crudo) - int((_dup | _nul | _val).sum()), "La zona curada debe quitar duplicados (conservando la primera aparición), tarjetas nulas y valores <= 0."
assert cuarentena["motivo"].value_counts().to_dict() == {"duplicado": int(_dup.sum()), "tarjeta_nula": int(_nul.sum()), "valor_invalido": int(_val.sum())}, "Cada motivo debe contar exactamente lo sembrado: usa la prioridad duplicado > tarjeta_nula > valor_invalido."
assert curada["id_viaje"].is_unique and curada["tarjeta"].notna().all() and (curada["valor_cop"] > 0).all(), "La zona curada debe ser única, completa y válida."
assert recibo == {"entran": len(crudo), "curada": len(curada), "cuarentena": len(cuarentena)} and recibo["entran"] == recibo["curada"] + recibo["cuarentena"], "El recibo debe cumplir entran = curada + cuarentena."
assert q_curada == {"completitud": 1.0, "unicidad": 1.0, "validez": 1.0}, "La zona curada debe tener completitud, unicidad y validez iguales a 1.0."
assert q_crudo is not None and set(q_crudo) == {"completitud", "unicidad", "validez"} and all(v < 1 for v in q_crudo.values()), "En la zona cruda las tres dimensiones de calidad deben ser menores que 1."
assert abs(q_crudo["unicidad"] - crudo["id_viaje"].nunique() / len(crudo)) < 1e-12, "unicidad = ids distintos / filas."
assert vigente is not None and archivo is not None and len(vigente) + len(archivo) == len(curada) and len(archivo) > 0 and len(vigente) > 0, "vigente + archivo debe conservar las filas de la zona curada, con filas en ambas."
assert archivo["fecha"].max() < HOY - pd.Timedelta(days=DIAS_RETENCION) <= vigente["fecha"].min(), "En archivo va lo anterior a hoy - dias; en vigente lo demás."
assert ps_a1 == ps_a2 and ps_a1 != ps_b and len(ps_a1) == 16 and TARJETA not in ps_a1, "El seudónimo debe ser determinista, depender de la sal y tener 16 caracteres sin mostrar la tarjeta."
assert ps_a1 == hashlib.sha256(("sal-A" + TARJETA).encode()).hexdigest()[:16], "ps = sha256(sal + tarjeta).hexdigest()[:16]."
assert isinstance(zonas, dict) and set(zonas) == {"cruda", "vigente", "archivo", "cuarentena"}, "zonas debe tener las llaves cruda, vigente, archivo y cuarentena."
_antes = sum(int((d["tarjeta"] == TARJETA).sum()) for d in zonas.values())
assert _antes > 0 and n_borradas == _antes, f"n_borradas debe ser el total de filas de {TARJETA} en todas las zonas ({_antes})."
assert all((d["tarjeta"] != TARJETA).all() for d in zonas_nuevas.values()), "Tras suprimir no puede quedar ninguna fila de la tarjeta en NINGUNA zona (incluida la cuarentena y el archivo)."
assert sum(len(d) for d in zonas_nuevas.values()) == sum(len(d) for d in zonas.values()) - n_borradas, "Suprimir no debe tocar las filas de otras tarjetas."
print(f"✅ Reto 4 superado: recibo {recibo} | calidad cruda {({k: round(float(v), 3) for k, v in q_crudo.items()})} | {n_borradas} filas de {TARJETA} borradas de las 4 zonas")

---
## Reto 5: Conclusiones 📝

**Puntaje sugerido:** 15 puntos (sin código). Escribe **5 a 8 líneas** en la celda de abajo interpretando *tus* resultados. Debe responder:

- ¿Por qué `tib_anio` es menor que `tb_anio` si son los mismos bytes? ¿En qué mes se llena el servidor y qué decisión tomarías: más disco, menos replicación o archivar? Usa tus cifras.
- ¿Qué te dice `techo` de Amdahl sobre añadir procesadores al trabajo nocturno y qué harías para **subir el techo** en vez de sumar procesadores?
- ¿Por qué el conteo por tiempo de llegada engaña? ¿Qué sacrificas al subir `L` (qué aprendiste de `perdidos` y del retraso p95)?
- En la unificación, ¿por qué un `NaN` de `pasajeros_afectados` **no** equivale a cero y por qué la cuarentena conserva las líneas malas en vez de descartarlas?
- ¿Por qué un *hash* de la tarjeta **sin sal** no basta para anonimizar y por qué hubo que borrar también del archivo y de la cuarentena?

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5-8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: volumen (SI frente a binario, replicación, crecimiento compuesto) y ley de Amdahl | 20 |
| Reto 2: ventanas por tiempo de evento y de llegada, marca de agua y pérdidas | 20 |
| Reto 3: lectura de CSV, JSON y texto a un esquema común, con cuarentena y conservación | 20 |
| Reto 4: validación, calidad, retención, seudonimización y supresión en todas las zonas | 25 |
| Reto 5: conclusiones argumentadas con tus cifras (5-8 líneas) | 15 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] Reinicié el kernel y ejecuté *Run All* sin errores.
- [ ] No quedan `# TODO` ni variables en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen ✅.
- [ ] En cada fuente, las filas válidas más las de cuarentena suman las de entrada.
- [ ] La supresión se aplicó a todas las zonas y no se tocaron otras tarjetas.
- [ ] Escribí mis conclusiones con cifras de mis propios resultados.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Big Data</i>
  </p>
</div>